In [0]:


storage_account_name = "energysentineldl"
storage_account_key = "OiRgE+cLxjUGAE2GvfkNRuiIV04cs6kvxHzKI3nVcOU4xE4PpYrWImVDp7Szdh/JGJJUFXyBKCW5+AStLuP97Q=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/")
print(" Connected!")


✅ Connected!


In [0]:

df = spark.read.csv(
    f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/training_data.csv",
    header=True,
    inferSchema=True
)

display(df)
print(f"Total rows: {df.count()}")
df.printSchema()

sensor_id,plant_id,timestamp,power_consumption,voltage,current,temperature,power_factor,frequency,is_anomaly
SENSOR_A01,PLANT_A,2026-06-28T22:26:46.878111Z,478.02,221.78,2.43,66.51,0.881,50.05,0
SENSOR_A02,PLANT_A,2026-06-28T22:26:46.878111Z,448.97,221.02,1.69,65.22,0.887,50.0,0
SENSOR_A03,PLANT_A,2026-06-28T22:26:46.878111Z,492.54,218.87,2.73,68.61,0.88,49.96,0
SENSOR_A04,PLANT_A,2026-06-28T22:26:46.878111Z,435.29,218.53,2.42,60.98,0.854,50.06,0
SENSOR_A05,PLANT_A,2026-06-28T22:26:46.878111Z,476.63,218.9,1.97,67.76,0.869,50.06,0
SENSOR_B01,PLANT_B,2026-06-28T22:26:46.878111Z,237.28,221.81,0.8,57.9,0.865,49.92,0
SENSOR_B02,PLANT_B,2026-06-28T22:26:46.878111Z,226.9,218.99,1.3,56.0,0.881,50.06,0
SENSOR_B03,PLANT_B,2026-06-28T22:26:46.878111Z,336.81,221.42,0.7,93.26,0.86,49.93,1
SENSOR_C01,PLANT_C,2026-06-28T22:26:46.878111Z,110.41,219.16,0.62,48.15,0.858,49.97,0
SENSOR_C02,PLANT_C,2026-06-28T22:26:46.878111Z,96.25,221.43,0.75,43.93,0.866,49.97,0


Total rows: 33850
root
 |-- sensor_id: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- power_consumption: double (nullable = true)
 |-- voltage: double (nullable = true)
 |-- current: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- frequency: double (nullable = true)
 |-- is_anomaly: integer (nullable = true)



In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType, IntegerType
)

BRONZE_PATH = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/training_data.csv"
SILVER_PATH = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/sensor_readings"
SILVER_QUARANTINE_PATH = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/sensor_readings_quarantine"

# ملاحظة: الاتصال بالـ ADLS متعمول already عن طريق spark.conf.set بالـ Access Key


# هنبدء نبني الاسكيما يدويا 

bronze_schema = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("plant_id", StringType(), True),
    StructField("timestamp", StringType(), True),   # هنعمله cast لـ Timestamp بعدين
    StructField("power_consumption", DoubleType(), True),
    StructField("voltage", DoubleType(), True),
    StructField("current", DoubleType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("power_factor", DoubleType(), True),
    StructField("frequency", DoubleType(), True),
    StructField("is_anomaly", IntegerType(), True),
])

# هنقرء الداتا من البرونز لير مع تعريف الاسكيما الجديد الي بنيناها يدويا 
df_bronze = (
    spark.read
    .option("header", "true")
    .schema(bronze_schema)
    .csv(BRONZE_PATH)
)

print(f"عدد الصفوف المقروءة من Bronze: {df_bronze.count()}")
df_bronze.printSchema()
display(df_bronze.limit(10))

# data quality checks 
# مرحله التحقق من جوده البيانات يدويا برضو بدون استخدام مكتبات 

# 1- Cast timestamp
df_checked = df_bronze.withColumn(
    "timestamp", F.to_timestamp("timestamp")
)

# 2- فحص القيم الأساسية المفقودة (nulls في أعمدة critical)
critical_cols = ["sensor_id", "plant_id", "timestamp", "power_consumption", "voltage"]

null_check_expr = F.lit(False)
for c in critical_cols:
    null_check_expr = null_check_expr | F.col(c).isNull()

df_checked = df_checked.withColumn("dq_has_null_critical", null_check_expr)

# 3- فحص المدى المنطقي للقيم (بناءً على الـ ranges الطبيعية + هامش للـ anomalies)
# القيم دي مبنية على specs المشروع: power_spike (1.8x-2.5x) و overheating (1.6x-2.0x)
df_checked = (
    df_checked
    .withColumn(
        "dq_voltage_out_of_range",
        (F.col("voltage") < 100) | (F.col("voltage") > 250)
    )
    .withColumn(
        "dq_temperature_out_of_range",
        (F.col("temperature") < 0) | (F.col("temperature") > 160)  # أعلى base_temp=72 * 2.0 = 144 + هامش
    )
    .withColumn(
        "dq_frequency_out_of_range",
        (F.col("frequency") < 45) | (F.col("frequency") > 60)  # الطبيعي حوالي 50Hz, fault يوصل 53-55
    )
    .withColumn(
        "dq_power_negative",
        F.col("power_consumption") < 0
    )
    .withColumn(
        "dq_power_factor_out_of_range",
        (F.col("power_factor") < 0) | (F.col("power_factor") > 1)
    )
)

# 4- فحص sensor_id غير معروف (لازم يكون من ضمن الـ 10 sensors)
valid_sensors = [
    "SENSOR_A01", "SENSOR_A02", "SENSOR_A03", "SENSOR_A04", "SENSOR_A05",
    "SENSOR_B01", "SENSOR_B02", "SENSOR_B03",
    "SENSOR_C01", "SENSOR_C02",
]
df_checked = df_checked.withColumn(
    "dq_unknown_sensor", ~F.col("sensor_id").isin(valid_sensors)
)

# 5- عمود إجمالي: الصف فيه أي مشكلة DQ؟
dq_flag_cols = [c for c in df_checked.columns if c.startswith("dq_")]
df_checked = df_checked.withColumn(
    "dq_is_valid",
    ~F.greatest(*[F.col(c).cast("int") for c in dq_flag_cols]).cast("boolean")
)

# تقرير سريع عن جوده البيانات 
total_rows = df_checked.count()
invalid_rows = df_checked.filter(~F.col("dq_is_valid")).count()

print(f"إجمالي الصفوف: {total_rows}")
print(f"صفوف بها مشاكل DQ: {invalid_rows} ({invalid_rows/total_rows*100:.2f}%)")

for c in dq_flag_cols:
    cnt = df_checked.filter(F.col(c)).count()
    if cnt > 0:
        print(f"  - {c}: {cnt} صف")

# بدايه عمليه الداتا كلين 

df_clean = (
    df_checked
    .filter(F.col("dq_is_valid"))
    .dropDuplicates(["sensor_id", "timestamp"])   # إزالة التكرار
    .drop(*dq_flag_cols, "dq_is_valid")
    .withColumn("silver_ingestion_ts", F.current_timestamp())
)

df_quarantine = (
    df_checked
    .filter(~F.col("dq_is_valid"))
    .withColumn("silver_ingestion_ts", F.current_timestamp())
)

print(f"صفوف Clean: {df_clean.count()}")
print(f"صفوف Quarantine: {df_quarantine.count()}")


numeric_cols = ["power_consumption", "voltage", "current", "temperature", "power_factor", "frequency"]

for c in numeric_cols:
    df_clean = df_clean.withColumn(c, F.round(F.col(c), 2))

display(df_clean.limit(10))


#  الكتابة إلى Silver كـ Delta


(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("plant_id")
    .save(SILVER_PATH)
)

(
    df_quarantine.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_QUARANTINE_PATH)
)

print("تم حفظ Silver + Quarantine بنجاح")




عدد الصفوف المقروءة من Bronze: 33850
root
 |-- sensor_id: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- power_consumption: double (nullable = true)
 |-- voltage: double (nullable = true)
 |-- current: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- frequency: double (nullable = true)
 |-- is_anomaly: integer (nullable = true)



sensor_id,plant_id,timestamp,power_consumption,voltage,current,temperature,power_factor,frequency,is_anomaly
SENSOR_A01,PLANT_A,2026-06-28T22:26:46.878111Z,478.02,221.78,2.43,66.51,0.881,50.05,0
SENSOR_A02,PLANT_A,2026-06-28T22:26:46.878111Z,448.97,221.02,1.69,65.22,0.887,50.0,0
SENSOR_A03,PLANT_A,2026-06-28T22:26:46.878111Z,492.54,218.87,2.73,68.61,0.88,49.96,0
SENSOR_A04,PLANT_A,2026-06-28T22:26:46.878111Z,435.29,218.53,2.42,60.98,0.854,50.06,0
SENSOR_A05,PLANT_A,2026-06-28T22:26:46.878111Z,476.63,218.9,1.97,67.76,0.869,50.06,0
SENSOR_B01,PLANT_B,2026-06-28T22:26:46.878111Z,237.28,221.81,0.8,57.9,0.865,49.92,0
SENSOR_B02,PLANT_B,2026-06-28T22:26:46.878111Z,226.9,218.99,1.3,56.0,0.881,50.06,0
SENSOR_B03,PLANT_B,2026-06-28T22:26:46.878111Z,336.81,221.42,0.7,93.26,0.86,49.93,1
SENSOR_C01,PLANT_C,2026-06-28T22:26:46.878111Z,110.41,219.16,0.62,48.15,0.858,49.97,0
SENSOR_C02,PLANT_C,2026-06-28T22:26:46.878111Z,96.25,221.43,0.75,43.93,0.866,49.97,0


إجمالي الصفوف: 33850
صفوف بها مشاكل DQ: 0 (0.00%)
صفوف Clean: 33850
صفوف Quarantine: 0


sensor_id,plant_id,timestamp,power_consumption,voltage,current,temperature,power_factor,frequency,is_anomaly,silver_ingestion_ts
SENSOR_B02,PLANT_B,2026-06-28T22:26:48.921064Z,218.24,219.28,1.26,53.72,0.87,50.03,0,2026-07-06T22:10:42.066661Z
SENSOR_A05,PLANT_A,2026-06-28T22:27:02.000753Z,477.42,218.36,2.57,65.55,0.86,50.08,0,2026-07-06T22:10:42.066661Z
SENSOR_A05,PLANT_A,2026-06-28T22:27:11.058987Z,492.96,221.7,1.74,67.74,0.85,49.94,0,2026-07-06T22:10:42.066661Z
SENSOR_A02,PLANT_A,2026-06-28T22:28:01.51202Z,449.41,219.71,1.89,62.62,0.88,50.02,0,2026-07-06T22:10:42.066661Z
SENSOR_A05,PLANT_A,2026-06-28T22:28:13.588046Z,481.89,220.29,1.74,66.68,0.85,50.01,0,2026-07-06T22:10:42.066661Z
SENSOR_C01,PLANT_C,2026-06-28T22:28:47.954019Z,106.19,221.32,0.19,46.33,0.88,49.97,0,2026-07-06T22:10:42.066661Z
SENSOR_A03,PLANT_A,2026-06-28T22:29:01.128208Z,484.89,220.72,2.4,66.41,0.86,50.06,0,2026-07-06T22:10:42.066661Z
SENSOR_A01,PLANT_A,2026-06-28T22:29:15.208933Z,483.95,219.32,1.99,64.72,0.88,50.01,0,2026-07-06T22:10:42.066661Z
SENSOR_B03,PLANT_B,2026-06-28T22:29:26.300289Z,250.26,221.06,0.76,59.82,0.87,50.09,0,2026-07-06T22:10:42.066661Z
SENSOR_A02,PLANT_A,2026-06-28T22:29:33.348946Z,451.45,221.4,2.29,66.44,0.88,50.09,0,2026-07-06T22:10:42.066661Z


✅ تم حفظ Silver + Quarantine بنجاح


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5828668271622017>, line 165
    152 (
    153     df_quarantine.write
    154     .format("delta")
   (...)
    157     .save(SILVER_QUARANTINE_PATH)
    158 )
    160 print("✅ تم حفظ Silver + Quarantine بنجاح")
--> 165 spark.sql(f"""
    166     CREATE TABLE IF NOT EXISTS silver_sensor_readings
    167     USING DELTA
    168     LOCATION '{SILVER_PATH}'
    169 """)
    171 spark.sql("SELECT plant_id, COUNT(*) as cnt FROM silver_sensor_readings GROUP BY plant_id").show()

File /databricks/spark/python/pyspark/sql/connect/session.py:879, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    876         _views.append(SubqueryAlias(df._plan, name))
    878 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 879 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    880 if "sql_command_result" in

In [0]:
df_silver = spark.read.format("delta").load(
    "abfss://silver@energysentineldl.dfs.core.windows.net/sensor_readings"
)
df_silver.show(10)

+----------+--------+--------------------+-----------------+-------+-------+-----------+------------+---------+----------+--------------------+
| sensor_id|plant_id|           timestamp|power_consumption|voltage|current|temperature|power_factor|frequency|is_anomaly| silver_ingestion_ts|
+----------+--------+--------------------+-----------------+-------+-------+-----------+------------+---------+----------+--------------------+
|SENSOR_C02| PLANT_C|2026-06-28 23:23:...|            98.82| 219.92|   0.72|      47.27|        0.89|    50.08|         0|2026-07-06 22:10:...|
|SENSOR_C01| PLANT_C|2026-06-28 23:22:...|           116.21|  219.1|   0.31|      50.64|        0.86|    49.94|         0|2026-07-06 22:10:...|
|SENSOR_C01| PLANT_C|2026-06-28 23:17:...|           121.49| 218.18|   0.43|      47.89|        0.86|    50.06|         0|2026-07-06 22:10:...|
|SENSOR_C02| PLANT_C|2026-06-28 23:14:...|            112.1| 218.49|   0.57|      47.22|        0.86|    50.07|         0|2026-07-06 22: